# Graph retrievers — interactive tour

Three `BaseRetriever`s over one AgensGraph store, the examples that teach the query
generator this engine's dialect, the index awareness that keeps a generated query off a
full label read, and the eval that holds all of it in place.

Every film, person and plot in this catalog is **invented**, so the model answering has
nothing to lean on but what each retriever hands it — every difference below is
retrieval, not recall.

**Prerequisites**

```bash
cd langchain
.venv/bin/python examples/demos/05_graph_retrievers/ingest.py    # this notebook's catalog
.venv/bin/python examples/demos/01_arxiv_graphrag/prepare.py     # optional, for the 138k-vertex leg
```

A handful of OpenAI calls run below (embeddings + `gpt-4o-mini`), together well under a cent.

In [1]:
import json
import pathlib
import sys

HERE = pathlib.Path.cwd()                       # .../05_graph_retrievers
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                                          # demos root, for _common
sys.path.insert(0, str(p.parent.parent / "evals" / "text2cypher"))   # the eval's own module

import pandas as pd
from _common import agens, config, models

from langchain_agensgraph import log_queries
from langchain_agensgraph.chains import DIALECT_EXAMPLES, AgensCypherQAChain
from langchain_agensgraph.retrievers import (
    AgensGraphContextRetriever,
    AgensText2CypherRetriever,
    AgensVectorRetriever,
    render_graph_context,
)

GRAPH = "movie_retrievers"
LIGHTHOUSE = "divers find a lighthouse that is somehow still burning"

conf = config.conf()
llm = models.get_llm()
store = agens.make_vector(models.get_embeddings(), graph_name=GRAPH,
                          node_label="Movie", text_node_property="plot")
graph = agens.make_graph(GRAPH, create=False)


def docs_df(docs, chars=46):
    return pd.DataFrame([{
        "title": d.metadata.get("title"),
        "year": d.metadata.get("year"),
        "score": round(d.metadata["score"], 3) if "score" in d.metadata else None,
        "neighbours": len(d.metadata.get("_context_nodes_", [])) or None,
        "edges": len(d.metadata.get("_context_rels_", [])) or None,
        "page_content": d.page_content[:chars].replace("\n", " ") + "…",
    } for d in docs])


print(f"{conf['host']}:{conf['port']}/{conf['dbname']} · graph {GRAPH}")

localhost:55432/agensgraph_demos · graph movie_retrievers


## The catalog

Eight films, their directors and cast, and their genres — movies carry the embedded plot
text, everyone else is ordinary graph structure around them.

In [2]:
vertices = graph.query("MATCH (n) RETURN label(n) AS label, count(*) AS n ORDER BY n DESC")
edges = graph.query("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY n DESC")
print("vertices:", {r["label"]: r["n"] for r in vertices})
print("edges:   ", {r["type"]: r["n"] for r in edges})

pd.DataFrame(graph.query(
    'MATCH (p:person)-[:directed]->(m:"Movie") '
    "RETURN m.title AS title, m.year AS year, p.name AS director ORDER BY year"))

vertices: {'person': 10, 'Movie': 8, 'genre': 6}
edges:    {'acted_in': 16, 'in_genre': 14, 'directed': 8}


,title,year,director
0,Winter Arithmetic,2017,Ines Varga
1,Marrow and Vine,2018,Sana Whitfield
2,The Salt Meridian,2019,Ines Varga
3,The Lantern Divers,2020,Sana Whitfield
4,Glasshouse Protocol,2021,Teodor Malik
5,The Eighth Ferry,2022,Ruth Okonjo
6,Quiet Cartography,2023,Teodor Malik
7,Static Bloom,2024,Ruth Okonjo


## 1 · `AgensVectorRetriever` — one similarity search, scored

The store's vector search (or its server-side hybrid fusion) as a LangChain retriever.
Each document carries its similarity score and the name of the retriever that produced it.

In [3]:
vector = AgensVectorRetriever(store=store, k=3)
docs_df(vector.invoke(LIGHTHOUSE))

,title,year,score,neighbours,edges,page_content
0,The Lantern Divers,2020,0.773,None,None,Deep-sea salvage divers find a sunken lighthou…
1,The Eighth Ferry,2022,0.322,None,None,A night-shift ferry pilot realizes her seven s…
2,Static Bloom,2024,0.252,None,None,A radio astronomer falls for the voice inside …


Every constructor value is a default a single call may override — `k`, `filter`, `params`,
`search_options`, `hybrid_config`, even the retrieval query.

In [4]:
print("constructor k=3 →", len(vector.invoke(LIGHTHOUSE)), "documents")
print("invoke k=1      →", len(vector.invoke(LIGHTHOUSE, k=1)), "document")

# A metadata filter reaches the database as a predicate, not a post-filter.
docs_df(vector.invoke("a quiet mystery", k=3, filter={"year": {"$gte": 2022}}))

constructor k=3 → 3 documents


invoke k=1      → 1 document


,title,year,score,neighbours,edges,page_content
0,Quiet Cartography,2023,0.353,None,None,A retired spy maps the silences in old wiretap…
1,Static Bloom,2024,0.254,None,None,A radio astronomer falls for the voice inside …
2,The Eighth Ferry,2022,0.236,None,None,A night-shift ferry pilot realizes her seven s…


## 2 · `AgensGraphContextRetriever` — the seed *and* its neighbourhood

A plot never names its director. This retriever answers the same similarity search and
brings each hit's graph neighbourhood back **in the same statement**, structured in
metadata: `_context_nodes_` and `_context_rels_`.

In [5]:
context = AgensGraphContextRetriever(store=store, k=1, expand_by_hops=1)
doc = context.invoke(LIGHTHOUSE)[0]
print(doc.page_content, "\n")

pd.DataFrame([{"label": n["label"],
               "name": n["properties"].get("name") or n["properties"].get("title")}
              for n in doc.metadata["_context_nodes_"]])

Deep-sea salvage divers find a sunken lighthouse whose lamp is somehow still burning. 



,label,name
0,person,Yusuf Adeyemi
1,person,Priya Nair
2,person,Sana Whitfield
3,genre,adventure
4,genre,mystery


`render_graph_context` is a ready-made `document_formatter` that writes that
neighbourhood under the seed's own text, for handing straight to a prompt.

In [6]:
rendered = AgensGraphContextRetriever(store=store, k=1, expand_by_hops=2,
                                      document_formatter=render_graph_context)
print(rendered.invoke(LIGHTHOUSE)[0].page_content)

Deep-sea salvage divers find a sunken lighthouse whose lamp is somehow still burning.

Graph context:
- Movie:The Salt Meridian
- Movie:Quiet Cartography
- Movie:Marrow and Vine
- Movie:The Eighth Ferry
- person:Yusuf Adeyemi
- person:Priya Nair
- person:Sana Whitfield
- genre:adventure
- genre:mystery
- person:Yusuf Adeyemi -[acted_in]-> Movie:The Salt Meridian
- person:Yusuf Adeyemi -[acted_in]-> Movie:The Lantern Divers
- person:Priya Nair -[acted_in]-> Movie:The Lantern Divers
- person:Sana Whitfield -[directed]-> Movie:The Lantern Divers
- person:Priya Nair -[acted_in]-> Movie:Marrow and Vine
- person:Sana Whitfield -[directed]-> Movie:Marrow and Vine
- person:Priya Nair -[acted_in]-> Movie:The Eighth Ferry
- Movie:The Lantern Divers -[in_genre]-> genre:adventure
- Movie:The Salt Meridian -[in_genre]-> genre:mystery
- Movie:The Lantern Divers -[in_genre]-> genre:mystery
- Movie:Quiet Cartography -[in_genre]-> genre:mystery
- Movie:Marrow and Vine -[in_genre]-> genre:mystery


Hops widen the reach; a single relationship type narrows it; the caps bound the payload
(they cut what is returned, not what is walked — which is why the hop count stops at 3).

In [7]:
rows = []
for hops in (1, 2, 3):
    d = AgensGraphContextRetriever(store=store, k=1, expand_by_hops=hops).invoke(LIGHTHOUSE)[0]
    rows.append({"setting": f"expand_by_hops={hops}",
                 "neighbours": len(d.metadata["_context_nodes_"]),
                 "edges": len(d.metadata["_context_rels_"])})

d = AgensGraphContextRetriever(store=store, k=1, expand_by_hops=2,
                               relationship_type="directed").invoke(LIGHTHOUSE)[0]
rows.append({"setting": 'hops=2 · relationship_type="directed"',
             "neighbours": len(d.metadata["_context_nodes_"]),
             "edges": len(d.metadata["_context_rels_"])})

d = AgensGraphContextRetriever(store=store, k=1, expand_by_hops=2, max_context_nodes=3,
                               max_context_rels=3).invoke(LIGHTHOUSE)[0]
rows.append({"setting": "hops=2 · caps=3", "neighbours": len(d.metadata["_context_nodes_"]),
             "edges": len(d.metadata["_context_rels_"])})

pd.DataFrame(rows)

,setting,neighbours,edges
0,expand_by_hops=1,5,5
1,expand_by_hops=2,9,12
2,expand_by_hops=3,19,20
3,"hops=2 · relationship_type=""directed""",2,2
4,hops=2 · caps=3,3,3


### One statement, whatever the fan-out

The expansion is appended to the seed search's own statement, so four seeds and every
neighbour within two hops arrive in one round trip — never one query per seed. The driver
reports every statement it sends, so the claim is checkable rather than promised: below,
exactly one record carries the retrieval. (A cold pooled connection also pays one-time
session setup — selecting the graph, caching the label table — and transaction control is
logged with no statement text at all.)

In [8]:
wide = AgensGraphContextRetriever(store=store, k=4, expand_by_hops=2)
wide.invoke(LIGHTHOUSE)                      # warm the pooled connection first

with log_queries() as sent:
    docs = wide.invoke(LIGHTHOUSE)

neighbours = sum(len(d.metadata["_context_nodes_"]) for d in docs)
print(f"{len(docs)} documents · {neighbours} neighbours · {len(sent)} record(s) logged\n")
display(pd.DataFrame([{
    "ms": round(one.elapsed * 1000, 2),
    "rows": one.rows,
    "carries the retrieval": "MATCH" in one.statement,
    "statement": one.statement[:52].replace("\n", " ").strip() or "(session control)",
} for one in sent]))

print(next(one.statement for one in sent if "MATCH" in one.statement))

4 documents · 43 neighbours · 2 record(s) logged



,ms,rows,carries the retrieval,statement
0,0.42,-1,False,(session control)
1,13.03,4,True,"MATCH (n:""Movie"") WITH n, n.""embe..."


MATCH (n:"Movie") 
            WITH n, n."embedding"::"vector"(1536)
            <=> %(embedding)s::"vector"(1536) AS
            inv_score ORDER BY inv_score LIMIT %(k)s WITH n, 1 - inv_score AS score
             WITH *, n as node 
    OPTIONAL MATCH (node)-[rels*1..2]-(peer)
    WITH node, score, id(node) AS seed_id, label(node) AS seed_label,
         collect(DISTINCT CASE WHEN peer IS NULL THEN NULL ELSE
             jsonb_build_object('id', id(peer), 'label', label(peer),
                 'properties', properties(peer) ||
                 jsonb_build_object('embedding', Null))
         END)[0..%(max_context_nodes)s] AS ctx_nodes,
         collect(DISTINCT CASE WHEN rels IS NULL THEN NULL ELSE
             jsonb_build_object('type', label(rels[-1]),
                 'start', start_id(rels[-1]), 'end', end_id(rels[-1]),
                 'properties', properties(rels[-1]))
         END)[0..%(max_context_rels)s] AS ctx_rels
    RETURN node."plot" AS text, score, node.__id__ AS doc_id

## 3 · What the graph context buys

The same question, the same model, two retrievers.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

ANSWER = ChatPromptTemplate.from_messages([
    ("system", "Answer from the context alone. If it does not contain the answer, say so plainly."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])


def answer(retriever, question):
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    return (ANSWER | llm).invoke({"context": context, "question": question}).content


question = "Who directed the film about divers finding a lighthouse that still burns?"
print("vector only     :", answer(AgensVectorRetriever(store=store, k=2), question))
print()
print("+ graph context :", answer(AgensGraphContextRetriever(
    store=store, k=2, expand_by_hops=2, document_formatter=render_graph_context), question))

vector only     : The context does not provide information about who directed the film about divers finding a lighthouse that still burns.



+ graph context : Sana Whitfield directed the film "The Lantern Divers," which is about divers finding a lighthouse that still burns.


## 4 · `AgensText2CypherRetriever` — the model writes the query, the server contains it

Counting is a query, not a similarity. The generated Cypher travels in each document's
metadata, so what ran is always visible.

In [10]:
scribe = AgensText2CypherRetriever(
    graph=graph, llm=llm, k=5,
    # The demo cluster's role is a superuser, which the read-only boundary refuses by
    # default; a deployment gives the model a plain role instead.
    allow_server_programs=True,
)
docs = scribe.invoke("How many films did each director make? Give the name and the count.")
print(docs[0].metadata["cypher"], "\n")
pd.DataFrame([json.loads(d.page_content) for d in docs])

MATCH (d:"person")-[r:"directed"]->(m:"Movie") 
RETURN d.name AS director, count(*) AS film_count 
ORDER BY film_count DESC LIMIT 5 



,director,film_count
0,Ines Varga,2
1,Ruth Okonjo,2
2,Sana Whitfield,2
3,Teodor Malik,2


### The boundary is the server's, not a regular expression

Reading the text is a fast first opinion — and it cannot be the boundary, because this is
PostgreSQL underneath, where `INSERT`, `TRUNCATE` and `COPY … TO PROGRAM` are all
available and none of them is Cypher. What actually refuses a write is a transaction the
server will not let write.

In [11]:
from langchain_agensgraph.chains.cypher_qa import is_write_query

for candidate in ('MATCH (m:"Movie") RETURN m.title',
                  'MATCH (m:"Movie") DETACH DELETE m',
                  'MATCH (m:"Movie") SET m.year = 1900'):
    print(f"{is_write_query(candidate)!s:>5}  ← is_write_query · {candidate}")

try:
    with graph.read_only(allow_server_programs=True):
        graph.query("CREATE (:\"Movie\" {title: 'Ghost Entry'})")
except Exception as exc:
    print("\nserver refused :", str(getattr(exc, "details", exc)).splitlines()[0][:150])
print("movies still   :", graph.query('MATCH (m:"Movie") RETURN count(*) AS n')[0]["n"])

False  ← is_write_query · MATCH (m:"Movie") RETURN m.title
 True  ← is_write_query · MATCH (m:"Movie") DETACH DELETE m
 True  ← is_write_query · MATCH (m:"Movie") SET m.year = 1900

server refused : cannot write to a graph in a read-only transaction (the server has no name for the command and reports it as '???')
movies still   : 8


### Self-correction, for the query that runs clean and matches nothing

A relationship written against its schema direction plans, runs, and returns zero rows.
There is no error to retry on, so `retry_on_empty` treats the silence itself as feedback
and asks again — measured live below against the 138k-vertex arXiv graph, where an
abstract never names its authors and `AUTHORED_BY` is the direction a model tends to
flip.

In [12]:
try:
    arxiv = agens.make_graph("arxiv", create=False)
    papers = arxiv.query('MATCH (p:"Paper") RETURN count(*) AS n')[0]["n"]
except Exception as exc:
    arxiv, papers = None, 0
    print("arxiv graph unavailable:", str(exc).splitlines()[0][:70])

if papers > 1000:
    print(f"{papers:,} papers\n")
    scholar = AgensText2CypherRetriever(graph=arxiv, llm=llm, k=3, max_retries=2,
                                        retry_on_empty=True, allow_server_programs=True)
    docs = scholar.invoke("Which author has written the most papers?")
    print(docs[0].metadata["cypher"], "\n")
    display(pd.DataFrame([json.loads(d.page_content) for d in docs]))
else:
    print("run examples/demos/01_arxiv_graphrag/prepare.py for this leg")

50,000 papers



MATCH (p:"Paper")-[r:"AUTHORED_BY"]->(a:"Author") 
RETURN a.name AS author, count(*) AS paper_count 
ORDER BY paper_count DESC LIMIT 3 



,author,paper_count
0,Chablat Damien IRCCyN,84
1,Aubert B.,66
2,The BABAR Collaboration,61


## 5 · Teaching the dialect by example

`DIALECT_EXAMPLES` is a pack of question-and-query pairs, each one there because a model
reaching for its habits gets that case wrong here — labels fold to lower case unless
double-quoted, `count()` takes an argument or `*`, a graph id is `labid.locid`,
`substring` starts at zero. Passed as `examples=`, they become real conversation turns
ahead of the question: each one shown answered with the query alone, which is the shape
the model is being asked to produce.

In [13]:
pd.set_option("display.max_colwidth", 78)
display(pd.DataFrame(DIALECT_EXAMPLES, columns=["question", "query"]).head(8))

taught = AgensCypherQAChain.from_llm(llm, graph=graph, top_k=10,
                                     examples=DIALECT_EXAMPLES, allow_server_programs=True)
turns = taught.cypher_prompt.format_messages(schema="…", question="…")
print(f"{len(turns)} messages · " + " → ".join(m.type for m in turns[:6]) + " → …")

,question,query
0,How many Product nodes are there?,"MATCH (p:""Product"") RETURN count(*) AS n LIMIT 1"
1,Who works at Acme?,"MATCH (p:""Person"")-[:""WORKS_AT""]->(c:""Company"") WHERE c.name = 'Acme' RETU..."
2,Which person is named exactly 'Ada'?,"MATCH (p:""Person"") WHERE p.name = 'Ada' RETURN p.name AS name LIMIT 10"
3,Which titles contain the word graph?,"MATCH (b:""Book"") WHERE b.title CONTAINS 'graph' RETURN b.title AS title LI..."
4,Who is older than 30?,"MATCH (p:""Person"") WHERE p.age > 30 RETURN p.name AS name, p.age AS age OR..."
5,Return the vertex whose graph id is 3.1.,MATCH (n) WHERE id(n) = 3.1 RETURN n LIMIT 1
6,"Which people are friends, in either direction?","MATCH (a:""Person"")-[:""FRIENDS_WITH""]-(b:""Person"") RETURN a.name AS a, b.na..."
7,Which authors wrote more than five papers?,"MATCH (a:""Author"")<-[:""AUTHORED_BY""]-(p:""Paper"") WITH a, count(*) AS paper..."


36 messages · system → human → ai → human → ai → human → …


## 6 · Index awareness

A predicate an index cannot serve returns the right rows and reads the whole label;
nothing but the plan says so. The first half of the answer is telling the model which
indexes exist — the schema it already reads now names them.

In [14]:
# Start from a graph with no property indexes, so this reads the same on a re-run.
for name in ("movie_title", "movie_year", "person_name"):
    graph.query(f"DROP PROPERTY INDEX IF EXISTS {name}")
graph.refresh_schema(force=True)
print("index section before:", "Property indexes" in graph.get_schema)

for ddl in ('CREATE PROPERTY INDEX IF NOT EXISTS movie_title ON "Movie" (title)',
            'CREATE PROPERTY INDEX IF NOT EXISTS movie_year ON "Movie" (year)',
            "CREATE PROPERTY INDEX IF NOT EXISTS person_name ON person (name)"):
    graph.query(ddl)
graph.refresh_schema(force=True)

print("index section after :", "Property indexes" in graph.get_schema, "\n")
lines = [ln.strip() for ln in graph.get_schema.strip().splitlines() if ln.strip()]
print("\n".join(lines[-2:]))

index section before: False
index section after : True 

Property indexes are the following (and id(x) is always indexed, on every label):
['(:"Movie") ON (title)', '(:"Movie") ON (year)', '(:"person") ON (name)']


### What those indexes can and cannot serve — measured here, now

`plan_has_index_path` (from the eval's own module) EXPLAINs a statement with
`enable_seqscan = off`: a plan that still carries `Disabled: true` is the planner's own
statement that **no index path exists at all** — as opposed to one it merely declined on
cost. Each pair below returns identical rows either way.

In [15]:
import t2c                       # the eval's module: dataset, result canon, plan probe

pairs = [
    ("prefix",
     "MATCH (m:\"Movie\") WHERE m.title STARTS WITH 'The' RETURN m.title AS title",
     "MATCH (m:\"Movie\") WHERE m.title >= 'The' AND m.title < 'Thf' RETURN m.title AS title"),
    ("arithmetic",
     'MATCH (m:"Movie") WHERE m.year + 0 > 2020 RETURN m.title AS title',
     'MATCH (m:"Movie") WHERE m.year > 2020 RETURN m.title AS title'),
    ("negation",
     'MATCH (m:"Movie") WHERE m.year <> 2019 RETURN m.title AS title',
     "MATCH (m:\"Movie\") WHERE m.year IN [2017, 2018, 2020, 2021, 2022, 2023, 2024] "
     "RETURN m.title AS title"),
]

rows = []
for case, habit, rewrite in pairs:
    for spelling, cypher in (("habit", habit), ("rewrite", rewrite)):
        rows.append({
            "case": case,
            "spelling": spelling,
            "rows": len(graph.query(cypher)),
            "index path": t2c.plan_has_index_path(conf, GRAPH, cypher),
            "predicate": cypher.split("WHERE ")[1].split(" RETURN")[0][:54],
        })
pd.DataFrame(rows)

,case,spelling,rows,index path,predicate
0,prefix,habit,3,False,m.title STARTS WITH 'The'
1,prefix,rewrite,3,True,m.title >= 'The' AND m.title < 'Thf'
2,arithmetic,habit,4,False,m.year + 0 > 2020
3,arithmetic,rewrite,4,True,m.year > 2020
4,negation,habit,7,False,m.year <> 2019
5,negation,rewrite,7,True,"m.year IN [2017, 2018, 2020, 2021, 2022, 2023, 2024]"


### Does the generator take the hint?

Three dialect-sensitive questions, generated with and without the example pack, judged by
the same plan probe. The rules live in the system prompt either way, so this is mostly a
check on *them* — and on where they still fall short: asked for "not 2019", both prompts
reach for `<>`, which answers correctly off a full label read. That is the one recurring
miss the eval reports too.

In [16]:
plain = AgensCypherQAChain.from_llm(llm, graph=graph, top_k=10, allow_server_programs=True)
questions = ["Which movie titles start with The?",
             "Which movies are not from 2019? Give their titles.",
             "Which film is titled exactly Static Bloom?"]

rows = []
for question in questions:
    for name, chain in (("0-shot", plain), ("few-shot", taught)):
        cypher = chain.generate_cypher(question)
        try:
            served = t2c.plan_has_index_path(conf, GRAPH, cypher)
        except Exception:
            served = None                     # generated something the planner refused
        rows.append({
            "question": question[:36],
            "prompt": name,
            "index path": served,
            "generated predicate": (cypher.split("WHERE ")[1][:52] if "WHERE " in cypher
                                    else cypher[:52]),
        })
pd.DataFrame(rows)

,question,prompt,index path,generated predicate
0,Which movie titles start with The?,0-shot,True,m.title >= 'The' AND m.title < 'Th' RETURN m.title A
1,Which movie titles start with The?,few-shot,True,m.title >= 'The' AND m.title < 'Thf' RETURN m.title
2,Which movies are not from 2019? Give,0-shot,False,m.year <> 2019 RETURN m.title AS title LIMIT 10
3,Which movies are not from 2019? Give,few-shot,False,m.year <> 2019 RETURN m.title AS title LIMIT 10
4,Which film is titled exactly Static,0-shot,True,"m.title = 'Static Bloom' RETURN m.title AS title, m."
5,Which film is titled exactly Static,few-shot,True,m.title = 'Static Bloom' RETURN m.title AS title LIM


## 7 · The eval that holds all of this in place

The dataset is 99 question-and-gold-query pairs in this dialect, most of them carrying
the query a model writes elsewhere together with the outcome that makes the trap real:
it errors, it returns different rows, or — the silent case — it returns *exactly* the
right rows off a full label read. None of that is asserted in prose: the gold queries,
the habit queries and the plan claims all execute.

In [17]:
dataset = t2c.load_dataset()
print(f"{len(dataset)} entries")
display(pd.Series([e["category"] for e in dataset]).value_counts().to_frame("entries").T)

# The dataset's own fixture graphs, built by the module that ships beside it.
for name in t2c.FIXTURE_GRAPHS:
    fixture = agens.make_graph(name, create=True, refresh_schema=False)
    t2c.BUILDERS[name](fixture)
    fixture.close()

rows = []
for entry in [e for e in dataset if e.get("habit_outcome") == "unindexed"]:
    fixture = agens.make_graph(entry["graph"], create=False, refresh_schema=False)
    gold, habit = fixture.query(entry["gold"]), fixture.query(entry["habit"])
    rows.append({
        "entry": entry["id"],
        "same rows": t2c.results_match(habit, gold, entry.get("ordered", False)),
        "gold index path": t2c.plan_has_index_path(conf, entry["graph"], entry["gold"]),
        "habit index path": t2c.plan_has_index_path(conf, entry["graph"], entry["habit"]),
    })
    fixture.close()
pd.DataFrame(rows)

99 entries


,gql,quoting,functions,jsonb,sargability,regex,aggregates,direction,paths,staging,arxiv,graphid
entries,16,12,12,10,9,7,7,6,6,6,5,3


,entry,same rows,gold index path,habit index path
0,sarg-arith-91,True,True,False
1,sarg-coalesce-92,True,True,False
2,sarg-prefix-93,True,True,False
3,sarg-neq-94,True,True,False
4,sarg-lower-95,True,True,False


### Scoring a model against it

```bash
cd langchain
.venv/bin/python evals/text2cypher/run.py --graphs t2c_movies,t2c_traps,arxiv
.venv/bin/python evals/text2cypher/run.py --examples 17 --retry --graphs t2c_movies,t2c_traps,arxiv
```

Measured on `gpt-4o-mini` over all 99 entries against 2.18-devel:

| configuration    | executable | execution accuracy | index-served |
| ---------------- | ---------- | ------------------ | ------------ |
| 0-shot           | 90%        | 60%                | 92%          |
| 0-shot + retry   | 94%        | 68%                | 92%          |
| 17-shot          | 83%        | 66%                | 92%          |
| 17-shot + retry  | 90%        | **73%**            | **92%**      |

The index rules and the schema's index section carry the index-served rate on their own —
it is 12/13 in every configuration, and the recurring miss is the honest kind: asked for
"not 2019", the model writes `<>`, which answers correctly off a full label read. Details
and the per-item files: `evals/text2cypher/README.md`.

In [18]:
store.close()
graph.close()
if arxiv is not None:
    arxiv.close()
agens.close()
print("connections closed")

connections closed
